# Repository guide: Main test comparison; Whisper; bootstrap; error analysis

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


# Final held-out evaluation — Tarifit ASR V1.3

Evaluates four reproducible trained CTC systems on the frozen 115-segment speaker-independent test set: MMS full + SpecAugment, MMS 50% Bible + SpecAugment, Fadhma full, and OmniASR full. It saves segment-level predictions, global WER/CER, CER without spaces, and per-speaker metrics. The notebook never edits the frozen references.

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 2 — Install evaluation dependencies
!pip install -q transformers accelerate jiwer soundfile safetensors sentencepiece scipy pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.5 MB/s eta 0:00:00


In [ ]:
# Cell 3 — Define project and evaluation paths
from pathlib import Path
PROJECT_ROOT=Path('/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm')
MANIFEST=PROJECT_ROOT/'data/metadata/segments_metadata_v1_3_final_eval.csv'
TOKENIZER_DIR=PROJECT_ROOT/'data/processed/mms_tokenizer_v1_2'
MODELS_DIR=PROJECT_ROOT/'models'
RESULTS_ROOT=PROJECT_ROOT/'results/final_test_v1_3'
RESULTS_ROOT.mkdir(parents=True,exist_ok=True)
print('Project:',PROJECT_ROOT.exists())
print('Manifest:',MANIFEST.exists())
print('Tokenizer:',TOKENIZER_DIR.exists())
print('Models:',MODELS_DIR.exists())
print('Results:',RESULTS_ROOT)


Project: True
Manifest: True
Tokenizer: True
Models: True
Results: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3


In [ ]:
# Cell 4 — Load and verify the frozen 115-segment test set
import pandas as pd
import numpy as np
df=pd.read_csv(MANIFEST)
test_df=df[df['dataset_split'].astype(str).str.lower().eq('test')].copy().reset_index(drop=True)
assert len(test_df)==115, f'Expected 115 test rows, found {len(test_df)}'
assert test_df['segment_id'].is_unique
assert test_df['transcription'].fillna('').astype(str).str.strip().ne('').all()
print('Segments:',len(test_df))
print('Duration min:',round(test_df['duration_seconds'].sum()/60,3))
print('Speakers:',test_df['speaker_group_id'].nunique())
print(test_df['speaker_group_id'].value_counts())


Segments: 115
Duration min: 17.921
Speakers: 2
speaker_group_id
SPK003    75
SPK004    40
Name: count, dtype: int64


In [ ]:
# Cell 5 — Resolve relative audio paths from the frozen manifest

from pathlib import Path
import pandas as pd

def resolve_audio_path(row):

    value = str(
        row["audio_path"]
    ).strip()

    # Absolute path already valid
    p = Path(value)

    if p.is_absolute() and p.exists():
        return str(p)

    # Relative project path
    p = PROJECT_ROOT / value

    if p.exists():
        return str(p)

    return None


test_df["resolved_audio_path"] = (
    test_df.apply(
        resolve_audio_path,
        axis=1
    )
)

missing = test_df[
    test_df["resolved_audio_path"].isna()
]

print(
    "Resolved:",
    len(test_df) - len(missing)
)

print(
    "Missing:",
    len(missing)
)

if len(missing) == 0:

    print(
        "\n✓ All 115 test audio files resolved."
    )

    print("\nExamples:")

    print(
        test_df[
            [
                "segment_id",
                "resolved_audio_path",
            ]
        ]
        .head()
        .to_string(index=False)
    )

else:

    print("\nFirst missing paths:")

    for _, row in missing.head(10).iterrows():

        expected = (
            PROJECT_ROOT
            / str(row["audio_path"])
        )

        print(
            row["segment_id"],
            "->",
            expected
        )

Resolved: 115
Missing: 0

✓ All 115 test audio files resolved.

Examples:
    segment_id                                                                                            resolved_audio_path
REC052_SEG0001 /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC052/REC052_SEG0001.wav
REC052_SEG0002 /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC052/REC052_SEG0002.wav
REC052_SEG0003 /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC052/REC052_SEG0003.wav
REC052_SEG0004 /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC052/REC052_SEG0004.wav
REC052_SEG0005 /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC052/REC052_SEG0005.wav


In [ ]:
# Cell 6 — Load the exact V1.2 tokenizer
from transformers import Wav2Vec2CTCTokenizer
tokenizer=Wav2Vec2CTCTokenizer.from_pretrained(str(TOKENIZER_DIR))
print('Tokenizer size:',len(tokenizer))
print('Pad:',tokenizer.pad_token,tokenizer.pad_token_id)
print('Delimiter:',tokenizer.word_delimiter_token)
assert len(tokenizer)==34


Tokenizer size: 34
Pad: [PAD] 33
Delimiter: |


In [ ]:
# Cell 7 — Discover saved model folders
print('Top-level model folders:')
for p in sorted(MODELS_DIR.iterdir()):
    if p.is_dir(): print('-',p.name)


Top-level model folders:
- fadhma_300m_tarifit_v1_1
- fadhma_300m_tarifit_v1_1_best
- fadhma_300m_tarifit_v1_1_best_backup
- fadhma_300m_tarifit_v1_2
- fadhma_300m_tarifit_v1_2_best
- fadhma_300m_tarifit_v1_2_best_backup
- fadhma_300m_tarifit_v1_2_quarter_bible
- fadhma_300m_tarifit_v1_2_transfer
- kabyle_xlsr_tarifit_v1
- kabyle_xlsr_tarifit_v1_best_backup
- kabyle_xlsr_transfer_v1_2_specaugment
- mms_1b_tarifit_v1_2_half_bible_specaug
- mms_1b_tarifit_v1_2_noaug
- mms_1b_tarifit_v1_2_specaug
- mms_1b_tarifit_v1_2_specaug_speed
- mms_M2_adapter_v1_1
- mms_M2b_tarifit_v1_1
- mms_tarifit_finetuned_v1_1
- omniASR_w2v_300m_tarifit_v1_2
- omniASR_w2v_300m_tarifit_v1_2_half_bible
- omniasr_w2v_300m_tarifit_v1
- whisper_small_corpus_v1
- xlsr_300m_X2_corpus_v1_1
- xlsr_300m_corpus_v1
- xlsr_300m_tarifit_v1_2_specaug


In [ ]:
# Cell 8 — Set the four final experiment roots explicitly

MODEL_ROOTS = {
    "mms_full_specaug": (
        MODELS_DIR
        / "mms_1b_tarifit_v1_2_specaug"
    ),

    "mms_half_bible_specaug": (
        MODELS_DIR
        / "mms_1b_tarifit_v1_2_half_bible_specaug"
    ),

    "fadhma_full": (
        MODELS_DIR
        / "fadhma_300m_tarifit_v1_2"
    ),

    "omni_full": (
        MODELS_DIR
        / "omniASR_w2v_300m_tarifit_v1_2"
    ),
}

for name, path in MODEL_ROOTS.items():
    print(
        f"{name:28s}",
        "->",
        path,
        "| exists:",
        path.exists(),
    )

assert all(
    path.exists()
    for path in MODEL_ROOTS.values()
)

resolved_roots = MODEL_ROOTS

mms_full_specaug             -> /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_specaug | exists: True
mms_half_bible_specaug       -> /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug | exists: True
fadhma_full                  -> /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_2 | exists: True
omni_full                    -> /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/omniASR_w2v_300m_tarifit_v1_2 | exists: True


In [ ]:
# Cell 9A — Inspect where the final model weights are stored

CHECK_DIRS = [
    MODELS_DIR / "mms_1b_tarifit_v1_2_specaug",
    MODELS_DIR / "mms_1b_tarifit_v1_2_half_bible_specaug",
    MODELS_DIR / "fadhma_300m_tarifit_v1_2",
    MODELS_DIR / "fadhma_300m_tarifit_v1_2_best",
    MODELS_DIR / "fadhma_300m_tarifit_v1_2_best_backup",
    MODELS_DIR / "omniASR_w2v_300m_tarifit_v1_2",
]

for root in CHECK_DIRS:

    print("\n" + "=" * 100)
    print(root.name)
    print("=" * 100)

    if not root.exists():
        print("DOES NOT EXIST")
        continue

    print("Root files:")
    for p in sorted(root.iterdir()):
        if p.is_file():
            print(" ", p.name)

    print("\nCheckpoints:")
    for cp in sorted(root.glob("checkpoint-*")):
        weights = (
            (cp / "model.safetensors").exists()
            or (cp / "pytorch_model.bin").exists()
        )

        print(
            cp.name,
            "| weights:",
            weights,
            "| trainer_state:",
            (cp / "trainer_state.json").exists()
        )


mms_1b_tarifit_v1_2_specaug
Root files:
  best_adapter.safetensors
  best_adapter_info.json

Checkpoints:

mms_1b_tarifit_v1_2_half_bible_specaug
Root files:

Checkpoints:
checkpoint-393 | weights: True | trainer_state: True
checkpoint-524 | weights: True | trainer_state: True

fadhma_300m_tarifit_v1_2
Root files:

Checkpoints:

fadhma_300m_tarifit_v1_2_best
Root files:

Checkpoints:

fadhma_300m_tarifit_v1_2_best_backup
Root files:

Checkpoints:

omniASR_w2v_300m_tarifit_v1_2
Root files:

Checkpoints:
checkpoint-660 | weights: True | trainer_state: True
checkpoint-880 | weights: True | trainer_state: True


In [ ]:
# Cell 9B — Resolve MMS adapter, Omni best checkpoint, and locate Fadhma weights

import json
from pathlib import Path
from safetensors.torch import load_file

# --------------------------------------------------
# 1. MMS full SpecAug adapter information
# --------------------------------------------------

mms_full_root = (
    MODELS_DIR
    / "mms_1b_tarifit_v1_2_specaug"
)

info_path = (
    mms_full_root
    / "best_adapter_info.json"
)

adapter_path = (
    mms_full_root
    / "best_adapter.safetensors"
)

print("=" * 100)
print("MMS FULL + SPECAUG")
print("=" * 100)

print("Adapter exists:", adapter_path.exists())
print("Info exists:", info_path.exists())

if info_path.exists():
    print("\nbest_adapter_info.json:")
    with open(
        info_path,
        "r",
        encoding="utf-8"
    ) as f:
        info = json.load(f)

    print(
        json.dumps(
            info,
            indent=2,
            ensure_ascii=False
        )
    )

if adapter_path.exists():

    state = load_file(
        str(adapter_path)
    )

    print("\nNumber of tensors:", len(state))

    print("\nFirst 30 parameter names:")
    for key in list(state.keys())[:30]:
        print(
            key,
            tuple(state[key].shape)
        )


# --------------------------------------------------
# 2. Omni checkpoint confirmation
# --------------------------------------------------

print("\n" + "=" * 100)
print("OMNI FULL")
print("=" * 100)

omni_root = (
    MODELS_DIR
    / "omniASR_w2v_300m_tarifit_v1_2"
)

for cp_name in [
    "checkpoint-660",
    "checkpoint-880",
]:

    cp = omni_root / cp_name

    state_path = (
        cp
        / "trainer_state.json"
    )

    print("\n", cp_name)

    if state_path.exists():

        with open(
            state_path,
            "r",
            encoding="utf-8"
        ) as f:
            state = json.load(f)

        print(
            "best_model_checkpoint:",
            state.get(
                "best_model_checkpoint"
            )
        )

        print(
            "best_metric:",
            state.get(
                "best_metric"
            )
        )

        history = state.get(
            "log_history",
            []
        )

        eval_rows = [
            x for x in history
            if "eval_cer" in x
        ]

        print("Last eval rows:")

        for row in eval_rows[-4:]:
            print(
                {
                    k: row.get(k)
                    for k in [
                        "epoch",
                        "eval_wer",
                        "eval_cer",
                        "eval_loss",
                    ]
                }
            )


# --------------------------------------------------
# 3. Search ONLY Fadhma model folders
# --------------------------------------------------

print("\n" + "=" * 100)
print("FADHMA V1.2 WEIGHT SEARCH")
print("=" * 100)

fadhma_dirs = [
    p
    for p in MODELS_DIR.iterdir()
    if (
        p.is_dir()
        and "fadhma" in p.name.lower()
        and "v1_2" in p.name.lower()
    )
]

for root in sorted(fadhma_dirs):

    print("\nROOT:", root.name)

    files = []

    for pattern in [
        "*.safetensors",
        "*.bin",
        "config.json",
        "trainer_state.json",
    ]:
        files.extend(
            root.rglob(pattern)
        )

    if not files:
        print("  No model files found.")
        continue

    for p in sorted(set(files)):
        print(
            " ",
            p.relative_to(root)
        )

MMS FULL + SPECAUG
Adapter exists: True
Info exists: True

best_adapter_info.json:
{
  "best_cer": 0.43267143661066004,
  "eval_wer": 0.8596938775510204,
  "eval_loss": 2.8712220191955566,
  "epoch": 3.0,
  "global_step": 330,
  "metadata_sha256": "4911f46e0a8c3656089677b8899d8296b9e1528d8aa3633a64728eb645f28fab",
  "base_model": "facebook/mms-1b-all",
  "tokenizer_size": 34,
  "augmentation": {
    "apply_spec_augment": true,
    "mask_time_prob": 0.05,
    "mask_time_length": 5,
    "mask_time_min_masks": 1,
    "mask_feature_prob": 0.0
  }
}

Number of tensors: 290

First 30 parameter names:
lm_head.bias (34,)
lm_head.weight (34, 1280)
wav2vec2.encoder.layers.0.adapter_layer.linear_1.bias (16,)
wav2vec2.encoder.layers.0.adapter_layer.linear_1.weight (16, 1280)
wav2vec2.encoder.layers.0.adapter_layer.linear_2.bias (1280,)
wav2vec2.encoder.layers.0.adapter_layer.linear_2.weight (1280, 16)
wav2vec2.encoder.layers.0.adapter_layer.norm.bias (1280,)
wav2vec2.encoder.layers.0.adapter_layer

In [ ]:
# Cell 9C — Verify Fadhma full V1.2 best checkpoint

import json

FADHMA_ROOT = (
    MODELS_DIR
    / "fadhma_300m_tarifit_v1_2_transfer"
)

for cp_name in [
    "checkpoint-330",
    "checkpoint-550",
]:

    path = (
        FADHMA_ROOT
        / cp_name
        / "trainer_state.json"
    )

    print("\n" + "=" * 80)
    print(cp_name)
    print("=" * 80)

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:
        state = json.load(f)

    print(
        "Best checkpoint:",
        state.get("best_model_checkpoint")
    )

    print(
        "Best metric:",
        state.get("best_metric")
    )

    for row in state.get(
        "log_history",
        []
    ):

        if "eval_cer" in row:

            print({
                "epoch": row.get("epoch"),
                "eval_wer": row.get("eval_wer"),
                "eval_cer": row.get("eval_cer"),
                "eval_loss": row.get("eval_loss"),
            })


checkpoint-330
Best checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_2_transfer/checkpoint-330
Best metric: 0.46249698528820643
{'epoch': 1.0, 'eval_wer': 1.003826530612245, 'eval_cer': 0.5227912211592571, 'eval_loss': 3.1148977279663086}
{'epoch': 2.0, 'eval_wer': 0.9332482993197279, 'eval_cer': 0.47495779403489025, 'eval_loss': 3.3170125484466553}
{'epoch': 3.0, 'eval_wer': 0.8962585034013606, 'eval_cer': 0.46249698528820643, 'eval_loss': 3.267947196960449}

checkpoint-550
Best checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_2_transfer/checkpoint-330
Best metric: 0.46249698528820643
{'epoch': 1.0, 'eval_wer': 1.003826530612245, 'eval_cer': 0.5227912211592571, 'eval_loss': 3.1148977279663086}
{'epoch': 2.0, 'eval_wer': 0.9332482993197279, 'eval_cer': 0.47495779403489025, 'eval_loss': 3.3170125484466553}
{'epoch': 3.0, 'eval_wer': 0.8962585034013606, 'eval_cer': 0.4624969852882

In [ ]:
# Cell 9D — Set exact final-test checkpoints explicitly

resolved_checkpoints = {
    "mms_half_bible_specaug": (
        MODELS_DIR
        / "mms_1b_tarifit_v1_2_half_bible_specaug"
        / "checkpoint-524"
    ),

    "mms_full_specaug": (
        MODELS_DIR
        / "mms_1b_tarifit_v1_2_specaug"
        / "best_adapter.safetensors"
    ),

    "fadhma_full": (
        MODELS_DIR
        / "fadhma_300m_tarifit_v1_2_transfer"
        / "checkpoint-330"
    ),

    "omni_full": (
        MODELS_DIR
        / "omniASR_w2v_300m_tarifit_v1_2"
        / "checkpoint-660"
    ),
}

for key, path in resolved_checkpoints.items():
    print(
        f"{key:28s}",
        "->",
        path,
        "| exists:",
        path.exists(),
    )

assert all(
    p.exists()
    for p in resolved_checkpoints.values()
)

mms_half_bible_specaug       -> /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug/checkpoint-524 | exists: True
mms_full_specaug             -> /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_specaug/best_adapter.safetensors | exists: True
fadhma_full                  -> /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/fadhma_300m_tarifit_v1_2_transfer/checkpoint-330 | exists: True
omni_full                    -> /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/omniASR_w2v_300m_tarifit_v1_2/checkpoint-660 | exists: True


In [ ]:
# Cell 10 — Define model metadata and evaluation helpers
import gc, math, re, unicodedata, soundfile as sf, torch
from scipy.signal import resample_poly
from jiwer import wer,cer
from transformers import AutoFeatureExtractor,Wav2Vec2ForCTC,Wav2Vec2Processor
MODEL_SPECS={
 'mms_full_specaug':{'base_model':'facebook/mms-1b-all','display_name':'MMS-1B full V1.2 + SpecAugment'},
 'mms_half_bible_specaug':{'base_model':'facebook/mms-1b-all','display_name':'MMS-1B 50% Bible + SpecAugment'},
 'fadhma_full':{'base_model':'agbalu/Fadhma-300M','display_name':'Fadhma-300M full V1.2'},
 'omni_full':{'base_model':'ylacombe/omniASR_W2V_300M_SSL','display_name':'OmniASR-W2V-300M full V1.2'},
}
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)
def norm_text(x): return re.sub(r'\s+',' ',unicodedata.normalize('NFC',str(x))).strip()
def load_audio(path):
    a,sr=sf.read(path,always_2d=False); a=np.asarray(a)
    if a.ndim==2: a=a.mean(axis=1)
    a=a.astype(np.float32)
    if sr!=16000:
        g=math.gcd(int(sr),16000); a=resample_poly(a,16000//g,int(sr)//g).astype(np.float32)
    return a
def global_metrics(refs,hyps):
    refs=[norm_text(x) for x in refs]; hyps=[norm_text(x) for x in hyps]
    return {'wer':float(wer(refs,hyps)),'cer':float(cer(refs,hyps)),'cer_no_spaces':float(cer([x.replace(' ','') for x in refs],[x.replace(' ','') for x in hyps]))}
# Cell 10B — Load standard checkpoints and MMS full adapter correctly

from safetensors.torch import load_file

def load_model(key):

    spec = MODEL_SPECS[key]

    fe = AutoFeatureExtractor.from_pretrained(
        spec["base_model"]
    )

    proc = Wav2Vec2Processor(
        feature_extractor=fe,
        tokenizer=tokenizer,
    )

    # ------------------------------------------------
    # Special case: MMS full was saved adapter-only
    # ------------------------------------------------

    if key == "mms_full_specaug":

        adapter_path = (
            resolved_checkpoints[
                "mms_full_specaug"
            ]
        )

        print(
            "Loading MMS base + saved best adapter:"
        )
        print(adapter_path)

        model = Wav2Vec2ForCTC.from_pretrained(
            "facebook/mms-1b-all",
            vocab_size=len(tokenizer),
            ignore_mismatched_sizes=True,
        )

        adapter_state = load_file(
            str(adapter_path)
        )

        print(
            "Adapter tensors:",
            len(adapter_state)
        )

        result = model.load_state_dict(
            adapter_state,
            strict=False,
        )

        print(
            "Unexpected adapter keys:",
            len(result.unexpected_keys)
        )

        # Critical safety check:
        # our saved adapter should contain the custom CTC head.
        has_head = any(
            "lm_head" in k
            for k in adapter_state
        )

        print(
            "Saved CTC head present:",
            has_head
        )

        assert has_head, (
            "best_adapter.safetensors does not contain "
            "the trained CTC head. Do not evaluate this "
            "model until we recover the head."
        )

    # ------------------------------------------------
    # Normal full checkpoints
    # ------------------------------------------------

    else:

        cp = resolved_checkpoints[key]

        print(
            "Loading checkpoint:",
            cp
        )

        model = (
            Wav2Vec2ForCTC
            .from_pretrained(
                str(cp)
            )
        )

    assert (
        model.config.vocab_size
        == len(tokenizer)
    ), (
        f"{key}: model vocab "
        f"{model.config.vocab_size} != "
        f"tokenizer {len(tokenizer)}"
    )

    model.to(DEVICE)
    model.eval()

    return proc, model

Device: cuda


In [ ]:
# Cell 11 — Define frozen-test evaluation for one model
from tqdm.auto import tqdm
@torch.inference_mode()
def evaluate_model(key):
    spec=MODEL_SPECS[key]; cp=resolved_checkpoints[key]
    print('\n'+'='*100); print(spec['display_name']); print('Checkpoint:',cp); print('='*100)
    proc,model=load_model(key); rows=[]
    for _,row in tqdm(test_df.iterrows(),total=len(test_df),desc=key):
        audio=load_audio(row['resolved_audio_path'])
        inp=proc(audio,sampling_rate=16000,return_tensors='pt')
        vals=inp.input_values.to(DEVICE); mask=getattr(inp,'attention_mask',None); mask=mask.to(DEVICE) if mask is not None else None
        if DEVICE=='cuda':
            with torch.autocast(device_type='cuda',dtype=torch.float16): out=model(input_values=vals,attention_mask=mask)
        else: out=model(input_values=vals,attention_mask=mask)
        ids=torch.argmax(out.logits,dim=-1); hyp=norm_text(tokenizer.batch_decode(ids.cpu().numpy())[0]); ref=norm_text(row['transcription'])
        rows.append({'segment_id':row['segment_id'],'recording_id':row.get('recording_id',''),'speaker_group_id':row.get('speaker_group_id',''),'duration_seconds':row['duration_seconds'],'reference':ref,'prediction':hyp,'segment_wer':float(wer(ref,hyp)),'segment_cer':float(cer(ref,hyp)),'prediction_empty':hyp=='','model_key':key,'model_name':spec['display_name'],'checkpoint':str(cp)})
    p=pd.DataFrame(rows); m=global_metrics(p.reference.tolist(),p.prediction.tolist())
    summary={**m,'model_key':key,'model_name':spec['display_name'],'checkpoint':str(cp),'segments':len(p),'duration_minutes':float(p.duration_seconds.sum()/60),'empty_predictions':int(p.prediction_empty.sum()),'wer_percent':m['wer']*100,'cer_percent':m['cer']*100,'cer_no_spaces_percent':m['cer_no_spaces']*100}
    srows=[]
    for sp,g in p.groupby('speaker_group_id'):
        sm=global_metrics(g.reference.tolist(),g.prediction.tolist()); srows.append({'speaker_group_id':sp,'segments':len(g),'duration_minutes':g.duration_seconds.sum()/60,'wer_percent':sm['wer']*100,'cer_percent':sm['cer']*100,'cer_no_spaces_percent':sm['cer_no_spaces']*100})
    sd=pd.DataFrame(srows); od=RESULTS_ROOT/key; od.mkdir(parents=True,exist_ok=True)
    p.to_csv(od/'predictions.csv',index=False,encoding='utf-8'); sd.to_csv(od/'per_speaker_metrics.csv',index=False,encoding='utf-8'); (od/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
    print(f"WER: {summary['wer_percent']:.2f}%"); print(f"CER: {summary['cer_percent']:.2f}%"); print(f"CER no spaces: {summary['cer_no_spaces_percent']:.2f}%"); print('Empty predictions:',summary['empty_predictions']); print('Saved:',od)
    del model,proc; gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary


In [ ]:
# Cell 11B — Smoke-test all four models on one frozen test segment

row = test_df.iloc[0]

audio = load_audio(
    row["resolved_audio_path"]
)

print(
    "Segment:",
    row["segment_id"]
)

print(
    "Reference:",
    row["transcription"]
)

for key in [
    "mms_half_bible_specaug",
    "fadhma_full",
    "omni_full",
    "mms_full_specaug",
]:

    print("\n" + "=" * 100)
    print(key)
    print("=" * 100)

    proc, model = load_model(key)

    inp = proc(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
    )

    vals = inp.input_values.to(
        DEVICE
    )

    mask = getattr(
        inp,
        "attention_mask",
        None,
    )

    if mask is not None:
        mask = mask.to(DEVICE)

    with torch.inference_mode():

        out = model(
            input_values=vals,
            attention_mask=mask,
        )

    ids = torch.argmax(
        out.logits,
        dim=-1,
    )

    hyp = tokenizer.batch_decode(
        ids.cpu().numpy()
    )[0]

    print(
        "Prediction:",
        norm_text(hyp)
    )

    del model, proc
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Segment: REC052_SEG0001
Reference: a neḍwer ɣa winaṭ ɣa tsemma ɣa lmuwḍuɛa n tzeddiɣt ɣa rekra ṭenniḍayi qa ḍini ljamɛiyyaṭ iqqenent ɣa krun a tɛawanent iwḍan ɣa winaṭ tbeddan kisen ḥma as nnafen rekra

mms_half_bible_specaug


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

Loading checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug/checkpoint-524


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Prediction: anedwer ɣa winat ɣa tsmaɣrmuwḍuan tzeddixt ɣar kra tenni ḍayiqadin ljamɛeyyaṭ iqqnent ɣakrun ad taɛawanent iwḍan ɣawinat tbeddankisen ḥma asnafen ri kra

fadhma_full


...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b-all
Key            | Status   |                                                                                            
---------------+----------+--------------------------------------------------------------------------------------------
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154]) vs model:torch.Size([34])            
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154, 1280]) vs model:torch.Size([34, 1280])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Adapter tensors: 290
Unexpected adapter keys: 0
Saved CTC head present: True
Prediction: anedwer ɣa winat ɣ tsmaɣ rmuwḍu an tzeddixt ɣar kra ṭenni dayiqadin ljamɛeyyat iqqen n t ɣakrun taɛawanentiwḍan ɣa winat ittbeddankisen ḥma asnafen rekra


In [ ]:
# Cell 12A — Evaluate MMS 50% Bible + SpecAugment

mms_half_summary = evaluate_model(
    "mms_half_bible_specaug"
)

mms_half_summary


MMS-1B 50% Bible + SpecAugment
Checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug/checkpoint-524
Loading checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug/checkpoint-524


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

mms_half_bible_specaug:   0%|          | 0/115 [00:00<?, ?it/s]

WER: 70.75%
CER: 18.97%
CER no spaces: 15.91%
Empty predictions: 0
Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/mms_half_bible_specaug


{'wer': 0.7075396825396826,
 'cer': 0.18965639390719094,
 'cer_no_spaces': 0.15909479077711358,
 'model_key': 'mms_half_bible_specaug',
 'model_name': 'MMS-1B 50% Bible + SpecAugment',
 'checkpoint': '/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug/checkpoint-524',
 'segments': 115,
 'duration_minutes': 17.921066666666665,
 'empty_predictions': 0,
 'wer_percent': 70.75396825396825,
 'cer_percent': 18.965639390719094,
 'cer_no_spaces_percent': 15.909479077711358}

In [ ]:
# Cell 12A-check — Inspect MMS half-Bible final-test predictions

preds = pd.read_csv(
    RESULTS_ROOT
    / "mms_half_bible_specaug"
    / "predictions.csv"
)

print(
    preds[
        [
            "segment_id",
            "speaker_group_id",
            "reference",
            "prediction",
            "segment_wer",
            "segment_cer",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

    segment_id speaker_group_id                                                                                                                                                                                                        reference                                                                                                                                                                                    prediction  segment_wer  segment_cer
REC052_SEG0001           SPK003                                           a neḍwer ɣa winaṭ ɣa tsemma ɣa lmuwḍuɛa n tzeddiɣt ɣa rekra ṭenniḍayi qa ḍini ljamɛiyyaṭ iqqenent ɣa krun a tɛawanent iwḍan ɣa winaṭ tbeddan kisen ḥma as nnafen rekra                                      anedwer ɣa winat ɣa tsmaɣrmuwḍuan tzeddixt ɣar kra tenni ḍayiqadin ljamɛeyyaṭ iqqnent ɣakrun ad taɛawanent iwḍan ɣawinat tbeddankisen ḥma asnafen ri kra     0.866667     0.192771
REC052_SEG0002           SPK003                                                   maca

In [ ]:
# Cell 12 — Run the four selected models on the frozen test set
MODEL_ORDER=['mms_half_bible_specaug','mms_full_specaug','fadhma_full','omni_full']
all_summaries=[]
for key in MODEL_ORDER: all_summaries.append(evaluate_model(key))
summary_df=pd.DataFrame(all_summaries)
summary_df.to_csv(RESULTS_ROOT/'final_model_comparison.csv',index=False,encoding='utf-8')
print('\n'+'='*100); print('FINAL MODEL COMPARISON'); print('='*100)
print(summary_df[['model_name','wer_percent','cer_percent','cer_no_spaces_percent','empty_predictions']].sort_values('cer_percent').to_string(index=False))
print('Saved:',RESULTS_ROOT/'final_model_comparison.csv')



MMS-1B 50% Bible + SpecAugment
Checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug/checkpoint-524
Loading checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_half_bible_specaug/checkpoint-524


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

mms_half_bible_specaug:   0%|          | 0/115 [00:00<?, ?it/s]

WER: 70.75%
CER: 18.97%
CER no spaces: 15.91%
Empty predictions: 0
Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/mms_half_bible_specaug

MMS-1B full V1.2 + SpecAugment
Checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_specaug/best_adapter.safetensors
Loading MMS base + saved best adapter:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_specaug/best_adapter.safetensors


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

...

fadhma_full:   0%|          | 0/115 [00:00<?, ?it/s]

WER: 74.48%
CER: 22.73%
CER no spaces: 19.85%
Empty predictions: 0
Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/fadhma_full

OmniASR-W2V-300M full V1.2
Checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/omniASR_w2v_300m_tarifit_v1_2/checkpoint-660
Loading checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/omniASR_w2v_300m_tarifit_v1_2/checkpoint-660


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

omni_full:   0%|          | 0/115 [00:00<?, ?it/s]

WER: 75.40%
CER: 22.66%
CER no spaces: 19.84%
Empty predictions: 0
Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/omni_full

FINAL MODEL COMPARISON
                    model_name  wer_percent  cer_percent  cer_no_spaces_percent  empty_predictions
MMS-1B 50% Bible + SpecAugment    70.753968    18.965639              15.909479                  0
MMS-1B full V1.2 + SpecAugment    73.134921    19.589090              16.660974                  0
    OmniASR-W2V-300M full V1.2    75.396825    22.663833              19.837746                  0
         Fadhma-300M full V1.2    74.484127    22.727595              19.854825                  0
Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/final_model_comparison.csv


In [ ]:
# Cell 13 — Show per-speaker results for all models
tabs=[]
for key in MODEL_ORDER:
    s=pd.read_csv(RESULTS_ROOT/key/'per_speaker_metrics.csv'); s.insert(0,'model_key',key); tabs.append(s)
all_speakers=pd.concat(tabs,ignore_index=True)
print(all_speakers[['model_key','speaker_group_id','segments','duration_minutes','wer_percent','cer_percent']].to_string(index=False))


             model_key speaker_group_id  segments  duration_minutes  wer_percent  cer_percent
mms_half_bible_specaug           SPK003        75         12.034400    76.085663    22.200855
mms_half_bible_specaug           SPK004        40          5.886667    60.071514    12.597266
      mms_full_specaug           SPK003        75         12.034400    78.584176    22.745726
      mms_full_specaug           SPK004        40          5.886667    62.216925    13.375394
           fadhma_full           SPK003        75         12.034400    79.476502    26.538462
           fadhma_full           SPK004        40          5.886667    64.481526    15.226078
             omni_full           SPK003        75         12.034400    81.618084    26.869658
             omni_full           SPK004        40          5.886667    62.932062    14.384858


In [ ]:
# Cell 14 — Inspect sample predictions without changing references
best=summary_df.sort_values('cer_percent').iloc[0]['model_key']; preds=pd.read_csv(RESULTS_ROOT/best/'predictions.csv')
print('Best by final-test CER:',best)
for _,r in preds.head(10).iterrows():
    print('\n'+'-'*100); print('SEGMENT:',r.segment_id); print('REFERENCE :',r.reference); print('PREDICTION:',r.prediction); print('WER:',round(r.segment_wer*100,2),'| CER:',round(r.segment_cer*100,2))
print('\nDo not modify frozen test references based on model predictions.')


Best by final-test CER: mms_half_bible_specaug

----------------------------------------------------------------------------------------------------
SEGMENT: REC052_SEG0001
REFERENCE : a neḍwer ɣa winaṭ ɣa tsemma ɣa lmuwḍuɛa n tzeddiɣt ɣa rekra ṭenniḍayi qa ḍini ljamɛiyyaṭ iqqenent ɣa krun a tɛawanent iwḍan ɣa winaṭ tbeddan kisen ḥma as nnafen rekra
PREDICTION: anedwer ɣa winat ɣa tsmaɣrmuwḍuan tzeddixt ɣar kra tenni ḍayiqadin ljamɛeyyaṭ iqqnent ɣakrun ad taɛawanent iwḍan ɣawinat tbeddankisen ḥma asnafen ri kra
WER: 86.67 | CER: 19.28

----------------------------------------------------------------------------------------------------
SEGMENT: REC052_SEG0002
REFERENCE : macanec mac twarix nec sme sseqsi ɣa esmaḥaḍ x zax ruxnec mac twarix nec manyaḍ ijjenn sma lḥella ixeddmm ḍiraḍas wahaw xeddem weḍḍ iragwac mant lḥaliɣawemsme
PREDICTION: macanc maymkttwarix nec itsme ac sseqsix ittsmaḥ aḍ xzaxrux nec mak twarix nec manaya ḍ ijjen sma lḥella ixeddem ḍ iraḍas wahawa ixeddem weḍ ḍi iraggʷ

In [ ]:
# Cell 15 — Load and verify the historical MMS greedy baseline

import torch
from transformers import AutoProcessor, AutoModelForCTC

BASELINE_ID = "iukocha/mms-tachebdant-from-tarifit"

baseline_processor = AutoProcessor.from_pretrained(
    BASELINE_ID
)

baseline_model = AutoModelForCTC.from_pretrained(
    BASELINE_ID
)

baseline_model.to(DEVICE)
baseline_model.eval()

print("Model:", BASELINE_ID)
print("Model vocab size:", baseline_model.config.vocab_size)
print("Tokenizer size:", len(baseline_processor.tokenizer))

row = test_df.iloc[0]

audio = load_audio(
    row["resolved_audio_path"]
)

inputs = baseline_processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt",
)

with torch.inference_mode():

    logits = baseline_model(
        inputs.input_values.to(DEVICE)
    ).logits

print("Logits shape:", logits.shape)
print(
    "Number of output classes:",
    logits.shape[-1]
)

pred_ids = torch.argmax(
    logits,
    dim=-1
)

raw_prediction = (
    baseline_processor
    .batch_decode(
        pred_ids.cpu()
    )[0]
)

print("\nREFERENCE:")
print(row["transcription"])

print("\nRAW BASELINE PREDICTION:")
print(raw_prediction)

processor_config.json:   0%|          | 0.00/299 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Model: iukocha/mms-tachebdant-from-tarifit
Model vocab size: 46
Tokenizer size: 46
Logits shape: torch.Size([1, 681, 46])
Number of output classes: 46

REFERENCE:
a neḍwer ɣa winaṭ ɣa tsemma ɣa lmuwḍuɛa n tzeddiɣt ɣa rekra ṭenniḍayi qa ḍini ljamɛiyyaṭ iqqenent ɣa krun a tɛawanent iwḍan ɣa winaṭ tbeddan kisen ḥma as nnafen rekra

RAW BASELINE PREDICTION:
a nədhwər ɣawinath ɣa tsmaɣ lmuwḍhuɛa n zəddixt ɣa rəšra thənni dhayə qa dhin ljamɛiyyath iqqənənt ɣa šrun, adtaɛawalən tiwdhan ɣawinath tbəddan kisən ḥmaas-nnafən rəšra.


In [ ]:
# Cell 15C — Audit baseline output characters against V1.2 alphabet

from collections import Counter
import unicodedata
import re
import pandas as pd

V12_ALPHABET = set(
    "a b c d ḍ e ɛ f g h ḥ i j k l m n p q r s t ṭ u v w x y z ɣ ʷ".split()
)

ALLOWED = V12_ALPHABET | {" "}

# Use historical validation raw predictions first
old = pd.read_csv(
    PROJECT_ROOT
    / "results"
    / "mms_tarifit"
    / "mms_tarifit_validation_predictions.csv"
)

char_counts = Counter(
    ch
    for text in old["prediction_raw"].fillna("").astype(str)
    for ch in text
)

outside = [
    (ch, count, unicodedata.name(ch, "UNKNOWN"))
    for ch, count in char_counts.items()
    if ch not in ALLOWED
]

outside = sorted(
    outside,
    key=lambda x: (-x[1], x[0])
)

print("Characters outside V1.2 alphabet:\n")

for ch, count, name in outside:
    print(
        repr(ch),
        "count=",
        count,
        "|",
        name
    )

Characters outside V1.2 alphabet:

'ə' count= 950 | LATIN SMALL LETTER SCHWA
'š' count= 265 | LATIN SMALL LETTER S WITH CARON
'.' count= 133 | FULL STOP
',' count= 119 | COMMA
'ṣ' count= 31 | LATIN SMALL LETTER S WITH DOT BELOW
'-' count= 29 | HYPHEN-MINUS
'ā' count= 12 | LATIN SMALL LETTER A WITH MACRON


In [ ]:
# Cell 15D — Final baseline-to-V1.3 normalization

import re
import unicodedata


def normalize_baseline_to_v13(text):

    text = unicodedata.normalize(
        "NFC",
        str(text).lower()
    )

    # Historical MMS/Tachebdant orthography
    text = (
        text
        .replace("ə", "e")
        .replace("š", "c")
    )

    # V1.2/V1.3 target orthography
    text = (
        text
        .replace("ṣ", "s")
        .replace("ẓ", "z")
        .replace("ṛ", "r")
        .replace("ǧ", "dj")
        .replace("ā", "a")
        .replace("3", "ɛ")
        .replace("o", "u")
    )

    # Punctuation is not part of the target transcription
    text = "".join(
        " "
        if unicodedata.category(ch).startswith("P")
        else ch
        for ch in text
    )

    # Standardize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return unicodedata.normalize(
        "NFC",
        text
    )

In [ ]:
# Cell 15E — Check normalized baseline prediction

normalized = normalize_baseline_to_v13(
    raw_prediction
)

print("REFERENCE:")
print(row["transcription"])

print("\nNORMALIZED BASELINE:")
print(normalized)

V13_ALLOWED = set(
    "a b c d ḍ e ɛ f g h ḥ i j k l m n p q r s t ṭ u v w x y z ɣ ʷ".split()
) | {" "}

bad = sorted(
    set(normalized) - V13_ALLOWED
)

print("\nCharacters outside V1.3 alphabet:", bad)

REFERENCE:
a neḍwer ɣa winaṭ ɣa tsemma ɣa lmuwḍuɛa n tzeddiɣt ɣa rekra ṭenniḍayi qa ḍini ljamɛiyyaṭ iqqenent ɣa krun a tɛawanent iwḍan ɣa winaṭ tbeddan kisen ḥma as nnafen rekra

NORMALIZED BASELINE:
a nedhwer ɣawinath ɣa tsmaɣ lmuwḍhuɛa n zeddixt ɣa recra thenni dhaye qa dhin ljamɛiyyath iqqenent ɣa crun adtaɛawalen tiwdhan ɣawinath tbeddan kisen ḥmaas nnafen recra

Characters outside V1.3 alphabet: []


In [ ]:
# Cell 16 — Evaluate MMS greedy baseline on frozen V1.3 test

import json
from tqdm.auto import tqdm
from jiwer import wer, cer

baseline_rows = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="MMS greedy baseline",
):

    audio = load_audio(
        row["resolved_audio_path"]
    )

    inputs = baseline_processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
    )

    with torch.inference_mode():

        logits = baseline_model(
            inputs.input_values.to(DEVICE)
        ).logits

    pred_ids = torch.argmax(
        logits,
        dim=-1,
    )

    prediction_raw = (
        baseline_processor
        .batch_decode(
            pred_ids.cpu()
        )[0]
    )

    prediction = normalize_baseline_to_v13(
        prediction_raw
    )

    reference = norm_text(
        row["transcription"]
    )

    baseline_rows.append({
        "segment_id":
            row["segment_id"],

        "speaker_group_id":
            row["speaker_group_id"],

        "duration_seconds":
            row["duration_seconds"],

        "reference":
            reference,

        "prediction_raw":
            prediction_raw,

        "prediction":
            prediction,

        "segment_wer":
            float(wer(reference, prediction)),

        "segment_cer":
            float(cer(reference, prediction)),
    })


baseline_df = pd.DataFrame(
    baseline_rows
)


# Global metrics

m = global_metrics(
    baseline_df["reference"].tolist(),
    baseline_df["prediction"].tolist(),
)

baseline_summary = {
    "model_name":
        "MMS Tachebdant greedy baseline",

    "model_id":
        BASELINE_ID,

    "decoder":
        "greedy CTC",

    "segments":
        len(baseline_df),

    "duration_minutes":
        float(
            baseline_df["duration_seconds"].sum()
            / 60
        ),

    "wer_percent":
        m["wer"] * 100,

    "cer_percent":
        m["cer"] * 100,

    "cer_no_spaces_percent":
        m["cer_no_spaces"] * 100,
}


# Per-speaker metrics

speaker_rows = []

for speaker, g in baseline_df.groupby(
    "speaker_group_id"
):

    sm = global_metrics(
        g["reference"].tolist(),
        g["prediction"].tolist(),
    )

    speaker_rows.append({
        "speaker_group_id":
            speaker,

        "segments":
            len(g),

        "duration_minutes":
            g["duration_seconds"].sum() / 60,

        "wer_percent":
            sm["wer"] * 100,

        "cer_percent":
            sm["cer"] * 100,

        "cer_no_spaces_percent":
            sm["cer_no_spaces"] * 100,
    })


baseline_speakers = pd.DataFrame(
    speaker_rows
)


# Save results

BASELINE_OUT = (
    RESULTS_ROOT
    / "mms_greedy_baseline"
)

BASELINE_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

baseline_df.to_csv(
    BASELINE_OUT / "predictions.csv",
    index=False,
    encoding="utf-8",
)

baseline_speakers.to_csv(
    BASELINE_OUT / "per_speaker_metrics.csv",
    index=False,
    encoding="utf-8",
)

with open(
    BASELINE_OUT / "summary.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        baseline_summary,
        f,
        ensure_ascii=False,
        indent=2,
    )


print("=" * 80)
print("MMS GREEDY BASELINE — FINAL TEST")
print("=" * 80)

print(
    f"WER: "
    f"{baseline_summary['wer_percent']:.2f}%"
)

print(
    f"CER: "
    f"{baseline_summary['cer_percent']:.2f}%"
)

print(
    f"CER no spaces: "
    f"{baseline_summary['cer_no_spaces_percent']:.2f}%"
)

print("\nPer speaker:")

print(
    baseline_speakers.to_string(
        index=False
    )
)

print(
    "\nSaved:",
    BASELINE_OUT
)

MMS greedy baseline:   0%|          | 0/115 [00:00<?, ?it/s]

MMS GREEDY BASELINE — FINAL TEST
WER: 66.98%
CER: 21.96%
CER no spaces: 21.55%

Per speaker:
speaker_group_id  segments  duration_minutes  wer_percent  cer_percent  cer_no_spaces_percent
          SPK003        75         12.034400    71.088638    24.369658              23.742584
          SPK004        40          5.886667    58.760429    17.223975              17.264914

Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/mms_greedy_baseline


Fine-tuning improved character-level transcription accuracy substantially, particularly when whitespace was excluded, but this improvement did not translate into lower word error rate. The zero-shot MMS baseline retained the best WER, whereas the balanced MMS fine-tuned system achieved the best CER.

In [ ]:
# Cell 17 — Compare baseline and best fine-tuned MMS segment by segment

import pandas as pd

BASELINE_PRED = (
    RESULTS_ROOT
    / "mms_greedy_baseline"
    / "predictions.csv"
)

FINETUNED_PRED = (
    RESULTS_ROOT
    / "mms_half_bible_specaug"
    / "predictions.csv"
)

base = pd.read_csv(BASELINE_PRED)
ft = pd.read_csv(FINETUNED_PRED)

comparison = base[
    [
        "segment_id",
        "speaker_group_id",
        "reference",
        "prediction",
        "segment_wer",
        "segment_cer",
    ]
].rename(
    columns={
        "prediction": "baseline_prediction",
        "segment_wer": "baseline_wer",
        "segment_cer": "baseline_cer",
    }
)

comparison = comparison.merge(
    ft[
        [
            "segment_id",
            "prediction",
            "segment_wer",
            "segment_cer",
        ]
    ].rename(
        columns={
            "prediction": "finetuned_prediction",
            "segment_wer": "finetuned_wer",
            "segment_cer": "finetuned_cer",
        }
    ),
    on="segment_id",
    validate="one_to_one",
)

comparison["cer_gain"] = (
    comparison["baseline_cer"]
    - comparison["finetuned_cer"]
)

comparison["wer_gain"] = (
    comparison["baseline_wer"]
    - comparison["finetuned_wer"]
)

print("Segments:", len(comparison))

print(
    "\nFine-tuned has lower CER:",
    (comparison["cer_gain"] > 0).sum()
)

print(
    "Baseline has lower CER:",
    (comparison["cer_gain"] < 0).sum()
)

print(
    "Equal CER:",
    (comparison["cer_gain"] == 0).sum()
)

print(
    "\nFine-tuned has lower WER:",
    (comparison["wer_gain"] > 0).sum()
)

print(
    "Baseline has lower WER:",
    (comparison["wer_gain"] < 0).sum()
)

print(
    "Equal WER:",
    (comparison["wer_gain"] == 0).sum()
)

OUT = (
    RESULTS_ROOT
    / "baseline_vs_best_finetuned.csv"
)

comparison.to_csv(
    OUT,
    index=False,
    encoding="utf-8",
)

print("\nSaved:", OUT)

Segments: 115

Fine-tuned has lower CER: 81
Baseline has lower CER: 28
Equal CER: 6

Fine-tuned has lower WER: 41
Baseline has lower WER: 60
Equal WER: 14

Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/baseline_vs_best_finetuned.csv


In [ ]:
# Cell 18 — Word-level error decomposition

from jiwer import process_words

def word_error_summary(df, prediction_col):

    result = process_words(
        df["reference"].tolist(),
        df[prediction_col].tolist(),
    )

    return {
        "hits": result.hits,
        "substitutions": result.substitutions,
        "deletions": result.deletions,
        "insertions": result.insertions,
        "wer": result.wer,
    }


baseline_errors = word_error_summary(
    comparison,
    "baseline_prediction"
)

finetuned_errors = word_error_summary(
    comparison,
    "finetuned_prediction"
)

print("BASELINE")
print(baseline_errors)

print("\nBEST FINE-TUNED MMS")
print(finetuned_errors)

BASELINE
{'hits': 867, 'substitutions': 1330, 'deletions': 323, 'insertions': 35, 'wer': 0.6698412698412698}

BEST FINE-TUNED MMS
{'hits': 766, 'substitutions': 1230, 'deletions': 524, 'insertions': 29, 'wer': 0.7075396825396826}


In [ ]:
# Cell 19 — Analyze word deletions in baseline vs fine-tuned MMS

from collections import Counter
from jiwer import process_words

def extract_deletions(refs, hyps):

    deleted = Counter()
    examples = []

    for segment_id, ref, hyp in zip(
        comparison["segment_id"],
        refs,
        hyps,
    ):

        result = process_words(
            ref,
            hyp,
        )

        ref_words = ref.split()
        hyp_words = hyp.split()

        for alignment in result.alignments[0]:

            if alignment.type != "delete":
                continue

            words = ref_words[
                alignment.ref_start_idx:
                alignment.ref_end_idx
            ]

            for word in words:
                deleted[word] += 1

            if words:
                examples.append({
                    "segment_id": segment_id,
                    "deleted_words": " ".join(words),
                    "reference": ref,
                    "prediction": hyp,
                })

    return deleted, pd.DataFrame(examples)


baseline_deleted, baseline_del_examples = extract_deletions(
    comparison["reference"].tolist(),
    comparison["baseline_prediction"].tolist(),
)

ft_deleted, ft_del_examples = extract_deletions(
    comparison["reference"].tolist(),
    comparison["finetuned_prediction"].tolist(),
)


print("=" * 80)
print("BASELINE — MOST DELETED WORDS")
print("=" * 80)

for word, count in baseline_deleted.most_common(30):
    print(f"{word:20s} {count}")


print("\n" + "=" * 80)
print("FINE-TUNED MMS — MOST DELETED WORDS")
print("=" * 80)

for word, count in ft_deleted.most_common(30):
    print(f"{word:20s} {count}")


baseline_del_examples.to_csv(
    RESULTS_ROOT / "baseline_deletion_examples.csv",
    index=False,
    encoding="utf-8",
)

ft_del_examples.to_csv(
    RESULTS_ROOT / "finetuned_deletion_examples.csv",
    index=False,
    encoding="utf-8",
)

BASELINE — MOST DELETED WORDS
n                    21
ḍi                   13
ijjen                6
u                    5
iwḍan                5
ggin                 5
wa                   5
zi                   5
ɣa                   4
i                    4
winaṭ                3
aḍ                   3
bu                   3
attas                3
nix                  3
ttubisaṭ             3
taman                3
xrun                 3
uktuber              3
an                   2
ṭraṭa                2
tsemma               2
bab                  2
ḍayes                2
faqat                2
ɣas                  2
mammec               2
x                    2
dji                  2
lla                  2

FINE-TUNED MMS — MOST DELETED WORDS
n                    27
ḍi                   15
ijjen                9
zi                   8
u                    8
ḍ                    7
i                    7
ad                   6
bu                   6
ɣa                   5
aḍ       

In [ ]:
# Cell 20 — Find segments where fine-tuning improves CER but worsens WER

interesting = comparison[
    (comparison["finetuned_cer"] < comparison["baseline_cer"])
    &
    (comparison["finetuned_wer"] > comparison["baseline_wer"])
].copy()

interesting["cer_improvement"] = (
    interesting["baseline_cer"]
    - interesting["finetuned_cer"]
)

interesting["wer_degradation"] = (
    interesting["finetuned_wer"]
    - interesting["baseline_wer"]
)

interesting = interesting.sort_values(
    "cer_improvement",
    ascending=False,
)

print(
    "Segments with better CER but worse WER:",
    len(interesting)
)

print()

for _, r in interesting.head(15).iterrows():

    print("=" * 100)
    print("SEGMENT:", r["segment_id"])

    print("\nREFERENCE:")
    print(r["reference"])

    print("\nBASELINE:")
    print(r["baseline_prediction"])

    print("\nFINE-TUNED:")
    print(r["finetuned_prediction"])

    print(
        "\nBaseline WER/CER:",
        round(r["baseline_wer"] * 100, 2),
        round(r["baseline_cer"] * 100, 2),
    )

    print(
        "Fine-tuned WER/CER:",
        round(r["finetuned_wer"] * 100, 2),
        round(r["finetuned_cer"] * 100, 2),
    )

Segments with better CER but worse WER: 34

SEGMENT: REC059_SEG0014

REFERENCE:
umi wa ssinen mecḥar i ɣa yeqqim ḍi ṭaddaṭ ḍ mermi i ɣa ygenfa ṭaddaṭ ṭacemratc marra tekkes agrawen id iggʷaren idja issuwjeḍ trump i lintixab

BASELINE:
wumi wer ssinen mecḥar i ɣa yeqqim di theddarth dh mer mi i ɣa yegenfa thadderththac mera ct marra tekksa grawen id igaren ija ssuwj dh ṭrump ilintixab

FINE-TUNED:
umi wassin n mecḥar iɣa iqqim ḍi ṭaddart ḍ mermi iɣa eggenfa taddarṭ tacemradc tmara tekks grawen i d iggʷaren ija issewjeḍ ttrump ilin tixab

Baseline WER/CER: 73.08 26.57
Fine-tuned WER/CER: 76.92 16.78
SEGMENT: REC052_SEG0102

REFERENCE:
winaṭ nix ḍ rfawakih tsema rxuḍaṭ nni ad tiri ṭiɣra kṭa minzi ruxa axeddam ḍi asya nix ḍi aferik

BASELINE:
winas nix dh lfawakih ṭsmer xuḍaḥ nni ad tiri thiɣrak ṭaran minzi ruxa axeddam dhi asya nix dhi waferik

FINE-TUNED:
winatnixḍ lfawakih tsmerxuḍah nni ad tiri tiɣra kta minzi ruxa axeddam ḍi asya nixḍiwafrik

Baseline WER/CER: 52.63 22.92
Fine-tuned W

In [ ]:
# Cell 21 — Load Whisper-small zero-shot baseline

import torch
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    pipeline,
)

WHISPER_ID = "openai/whisper-small"

whisper_dtype = (
    torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

whisper_model = (
    AutoModelForSpeechSeq2Seq
    .from_pretrained(
        WHISPER_ID,
        torch_dtype=whisper_dtype,
        low_cpu_mem_usage=True,
    )
)

whisper_model.to(DEVICE)
whisper_model.eval()

whisper_processor = (
    AutoProcessor.from_pretrained(
        WHISPER_ID
    )
)

whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=whisper_dtype,
    device=0 if DEVICE == "cuda" else -1,
    generate_kwargs={
        "task": "transcribe"
    },
)

print("Model:", WHISPER_ID)
print("Device:", DEVICE)
print("Language forced: NO")
print("Task: transcribe")

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Model: openai/whisper-small
Device: cuda
Language forced: NO
Task: transcribe


In [ ]:
# Cell 22 — Smoke-test Whisper on one frozen test segment

row = test_df.iloc[0]

audio = load_audio(
    row["resolved_audio_path"]
)

result = whisper_pipe(
    {
        "array": audio,
        "sampling_rate": 16000,
    }
)

print("SEGMENT:")
print(row["segment_id"])

print("\nREFERENCE:")
print(row["transcription"])

print("\nWHISPER:")
print(result["text"])

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

SEGMENT:
REC052_SEG0001

REFERENCE:
a neḍwer ɣa winaṭ ɣa tsemma ɣa lmuwḍuɛa n tzeddiɣt ɣa rekra ṭenniḍayi qa ḍini ljamɛiyyaṭ iqqenent ɣa krun a tɛawanent iwḍan ɣa winaṭ tbeddan kisen ḥma as nnafen rekra

WHISPER:
 أنظور غاوينة تسمغ الموضوع أن تدخل غاوينة تأتب الدنكي سنحمغ سنفن رشرة


In [ ]:
# Cell 23 — Define Whisper script classification and evaluation normalization

import re
import unicodedata


def is_arabic_char(ch):

    cp = ord(ch)

    return (
        0x0600 <= cp <= 0x06FF
        or 0x0750 <= cp <= 0x077F
        or 0x08A0 <= cp <= 0x08FF
        or 0xFB50 <= cp <= 0xFDFF
        or 0xFE70 <= cp <= 0xFEFF
    )


def classify_script(text):

    text = str(text)

    if not text.strip():
        return "empty"

    arabic = 0
    latin = 0
    other_letters = 0

    for ch in text:

        if not ch.isalpha():
            continue

        if is_arabic_char(ch):
            arabic += 1

        elif "LATIN" in unicodedata.name(
            ch,
            ""
        ):
            latin += 1

        else:
            other_letters += 1

    total = (
        arabic
        + latin
        + other_letters
    )

    if total == 0:
        return "other"

    arabic_ratio = arabic / total
    latin_ratio = latin / total

    if arabic_ratio >= 0.8:
        return "arabic"

    if latin_ratio >= 0.8:
        return "latin"

    return "mixed"


def normalize_whisper_for_eval(text):

    text = unicodedata.normalize(
        "NFC",
        str(text).lower()
    )

    # Only mappings already justified
    # by the V1.2/V1.3 convention.
    text = (
        text
        .replace("ə", "e")
        .replace("š", "c")
        .replace("ṣ", "s")
        .replace("ẓ", "z")
        .replace("ṛ", "r")
        .replace("ǧ", "dj")
        .replace("3", "ɛ")
        .replace("o", "u")
    )

    # Remove punctuation, not letters.
    text = "".join(
        " "
        if unicodedata.category(ch).startswith("P")
        else ch
        for ch in text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

In [ ]:
# Cell 24 — Evaluate Whisper zero-shot on the frozen final test

from tqdm.auto import tqdm
from jiwer import wer, cer
import pandas as pd
import json

whisper_rows = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Whisper-small zero-shot",
):

    audio = load_audio(
        row["resolved_audio_path"]
    )

    result = whisper_pipe(
        {
            "array": audio,
            "sampling_rate": 16000,
        }
    )

    raw_prediction = (
        result["text"]
        if isinstance(result, dict)
        else str(result)
    )

    prediction = (
        normalize_whisper_for_eval(
            raw_prediction
        )
    )

    reference = norm_text(
        row["transcription"]
    )

    script = classify_script(
        raw_prediction
    )

    whisper_rows.append({
        "segment_id":
            row["segment_id"],

        "speaker_group_id":
            row["speaker_group_id"],

        "duration_seconds":
            row["duration_seconds"],

        "reference":
            reference,

        "prediction_raw":
            raw_prediction,

        "prediction_normalized":
            prediction,

        "script":
            script,

        "segment_wer":
            float(
                wer(
                    reference,
                    prediction
                )
            ),

        "segment_cer":
            float(
                cer(
                    reference,
                    prediction
                )
            ),
    })


whisper_df = pd.DataFrame(
    whisper_rows
)


# --------------------------------------------------
# Script distribution
# --------------------------------------------------

script_counts = (
    whisper_df["script"]
    .value_counts()
)

script_percent = (
    whisper_df["script"]
    .value_counts(
        normalize=True
    )
    .mul(100)
)


# --------------------------------------------------
# Descriptive whole-test WER/CER
# --------------------------------------------------
# These are saved, but interpretation is limited
# when Whisper outputs another script.

whisper_metrics = global_metrics(
    whisper_df["reference"].tolist(),
    whisper_df[
        "prediction_normalized"
    ].tolist(),
)


whisper_summary = {
    "model":
        WHISPER_ID,

    "mode":
        "zero-shot automatic language detection",

    "segments":
        len(whisper_df),

    "duration_minutes":
        float(
            whisper_df[
                "duration_seconds"
            ].sum() / 60
        ),

    "wer_percent":
        whisper_metrics["wer"] * 100,

    "cer_percent":
        whisper_metrics["cer"] * 100,

    "cer_no_spaces_percent":
        whisper_metrics[
            "cer_no_spaces"
        ] * 100,

    "script_counts":
        script_counts.to_dict(),

    "script_percent":
        script_percent.to_dict(),
}


# --------------------------------------------------
# Save
# --------------------------------------------------

WHISPER_OUT = (
    RESULTS_ROOT
    / "whisper_small_zero_shot"
)

WHISPER_OUT.mkdir(
    parents=True,
    exist_ok=True
)

whisper_df.to_csv(
    WHISPER_OUT
    / "predictions.csv",
    index=False,
    encoding="utf-8",
)

with open(
    WHISPER_OUT
    / "summary.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        whisper_summary,
        f,
        ensure_ascii=False,
        indent=2,
    )


print("=" * 80)
print("WHISPER-SMALL ZERO-SHOT — FINAL TEST")
print("=" * 80)

print("\nScript distribution:")
print(script_counts)

print("\nPercent:")
print(
    script_percent.round(2)
)

print(
    "\nDescriptive WER:",
    f"{whisper_summary['wer_percent']:.2f}%"
)

print(
    "Descriptive CER:",
    f"{whisper_summary['cer_percent']:.2f}%"
)

print(
    "CER no spaces:",
    f"{whisper_summary['cer_no_spaces_percent']:.2f}%"
)

print(
    "\nSaved:",
    WHISPER_OUT
)

Whisper-small zero-shot:   0%|          | 0/115 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


WHISPER-SMALL ZERO-SHOT — FINAL TEST

Script distribution:
script
arabic    110
latin       4
mixed       1
Name: count, dtype: int64

Percent:
script
arabic    95.65
latin      3.48
mixed      0.87
Name: proportion, dtype: float64

Descriptive WER: 187.90%
Descriptive CER: 166.18%
CER no spaces: 175.13%

Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/whisper_small_zero_shot


In [ ]:
# Cell 25 — Inspect representative Whisper outputs by script

for script in [
    "arabic",
    "latin",
    "mixed",
    "other",
]:

    subset = whisper_df[
        whisper_df["script"] == script
    ]

    if subset.empty:
        continue

    print("\n" + "=" * 100)
    print(
        script.upper(),
        "-",
        len(subset),
        "segments"
    )
    print("=" * 100)

    for _, r in subset.head(5).iterrows():

        print(
            "\nSEGMENT:",
            r["segment_id"]
        )

        print(
            "REFERENCE:",
            r["reference"]
        )

        print(
            "WHISPER:",
            r["prediction_raw"]
        )


ARABIC - 110 segments

SEGMENT: REC052_SEG0001
REFERENCE: a neḍwer ɣa winaṭ ɣa tsemma ɣa lmuwḍuɛa n tzeddiɣt ɣa rekra ṭenniḍayi qa ḍini ljamɛiyyaṭ iqqenent ɣa krun a tɛawanent iwḍan ɣa winaṭ tbeddan kisen ḥma as nnafen rekra
WHISPER:  أنظور غاوينة تسمغ الموضوع أن تدخل غاوينة تأتب الدنكي سنحمغ سنفن رشرة

SEGMENT: REC052_SEG0002
REFERENCE: macanec mac twarix nec sme sseqsi ɣa esmaḥaḍ x zax ruxnec mac twarix nec manyaḍ ijjenn sma lḥella ixeddmm ḍiraḍas wahaw xeddem weḍḍ iragwac mant lḥaliɣawemsme
WHISPER:  لا يجب أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن تقوم بإمكانك أن 

In [ ]:
# Cell 26 — Bootstrap 95% confidence intervals on frozen final test

import numpy as np
import pandas as pd
from jiwer import wer, cer

rng = np.random.default_rng(42)

MODEL_FILES = {
    "MMS baseline": (
        RESULTS_ROOT
        / "mms_greedy_baseline"
        / "predictions.csv"
    ),
    "MMS 50% Bible + SpecAug": (
        RESULTS_ROOT
        / "mms_half_bible_specaug"
        / "predictions.csv"
    ),
    "MMS full + SpecAug": (
        RESULTS_ROOT
        / "mms_full_specaug"
        / "predictions.csv"
    ),
    "Fadhma full": (
        RESULTS_ROOT
        / "fadhma_full"
        / "predictions.csv"
    ),
    "OmniASR full": (
        RESULTS_ROOT
        / "omni_full"
        / "predictions.csv"
    ),
}

predictions = {}

for name, path in MODEL_FILES.items():

    x = pd.read_csv(path)

    # Standardize prediction-column naming
    if "prediction" not in x.columns:
        raise ValueError(
            f"{name}: prediction column missing"
        )

    predictions[name] = x


def bootstrap_indices_stratified(df, rng):

    sampled = []

    for speaker, group in df.groupby(
        "speaker_group_id"
    ):

        idx = group.index.to_numpy()

        sampled_idx = rng.choice(
            idx,
            size=len(idx),
            replace=True,
        )

        sampled.extend(
            sampled_idx.tolist()
        )

    return sampled


def compute_metrics(df):

    refs = (
        df["reference"]
        .astype(str)
        .tolist()
    )

    hyps = (
        df["prediction"]
        .astype(str)
        .tolist()
    )

    refs_ns = [
        x.replace(" ", "")
        for x in refs
    ]

    hyps_ns = [
        x.replace(" ", "")
        for x in hyps
    ]

    return (
        wer(refs, hyps),
        cer(refs, hyps),
        cer(refs_ns, hyps_ns),
    )


N_BOOT = 5000

results = []

for name, df_model in predictions.items():

    boot = []

    for _ in range(N_BOOT):

        idx = bootstrap_indices_stratified(
            df_model,
            rng,
        )

        sample = df_model.loc[idx]

        boot.append(
            compute_metrics(sample)
        )

    boot = np.asarray(boot)

    point = compute_metrics(
        df_model
    )

    for metric_idx, metric_name in enumerate(
        [
            "WER",
            "CER",
            "CER_no_spaces",
        ]
    ):

        low, high = np.percentile(
            boot[:, metric_idx],
            [2.5, 97.5],
        )

        results.append({
            "model": name,
            "metric": metric_name,

            "estimate_percent":
                point[metric_idx] * 100,

            "ci_low_percent":
                low * 100,

            "ci_high_percent":
                high * 100,
        })


bootstrap_df = pd.DataFrame(
    results
)

bootstrap_df.to_csv(
    RESULTS_ROOT
    / "bootstrap_95ci_final_test.csv",
    index=False,
    encoding="utf-8",
)

print(
    bootstrap_df.to_string(
        index=False
    )
)

                  model        metric  estimate_percent  ci_low_percent  ci_high_percent
           MMS baseline           WER         66.984127       64.427114        69.520559
           MMS baseline           CER         21.962451       20.800663        23.195513
           MMS baseline CER_no_spaces         21.554227       20.339978        22.811552
MMS 50% Bible + SpecAug           WER         70.753968       68.742450        72.812161
MMS 50% Bible + SpecAug           CER         18.965639       18.053834        19.954372
MMS 50% Bible + SpecAug CER_no_spaces         15.909479       14.982977        16.901930
     MMS full + SpecAug           WER         73.134921       71.017400        75.191935
     MMS full + SpecAug           CER         19.589090       18.668442        20.525865
     MMS full + SpecAug CER_no_spaces         16.660974       15.783039        17.588888
            Fadhma full           WER         74.484127       72.468972        76.475332
            Fadhma fu

In [ ]:
# Cell 27 — Paired bootstrap: baseline vs best fine-tuned MMS

base = pd.read_csv(
    RESULTS_ROOT
    / "mms_greedy_baseline"
    / "predictions.csv"
)

ft = pd.read_csv(
    RESULTS_ROOT
    / "mms_half_bible_specaug"
    / "predictions.csv"
)

paired = base[
    [
        "segment_id",
        "speaker_group_id",
        "reference",
        "prediction",
    ]
].rename(
    columns={
        "prediction":
            "baseline_prediction"
    }
).merge(
    ft[
        [
            "segment_id",
            "prediction",
        ]
    ].rename(
        columns={
            "prediction":
                "finetuned_prediction"
        }
    ),
    on="segment_id",
    validate="one_to_one",
)

rng = np.random.default_rng(42)

diffs = []

for _ in range(5000):

    sampled = []

    for speaker, group in paired.groupby(
        "speaker_group_id"
    ):

        idx = group.index.to_numpy()

        sampled.extend(
            rng.choice(
                idx,
                size=len(idx),
                replace=True,
            ).tolist()
        )

    s = paired.loc[sampled]

    refs = s["reference"].tolist()

    base_hyp = (
        s["baseline_prediction"]
        .tolist()
    )

    ft_hyp = (
        s["finetuned_prediction"]
        .tolist()
    )

    base_wer = wer(
        refs,
        base_hyp
    )

    ft_wer = wer(
        refs,
        ft_hyp
    )

    base_cer = cer(
        refs,
        base_hyp
    )

    ft_cer = cer(
        refs,
        ft_hyp
    )

    refs_ns = [
        x.replace(" ", "")
        for x in refs
    ]

    base_ns = [
        x.replace(" ", "")
        for x in base_hyp
    ]

    ft_ns = [
        x.replace(" ", "")
        for x in ft_hyp
    ]

    base_cer_ns = cer(
        refs_ns,
        base_ns
    )

    ft_cer_ns = cer(
        refs_ns,
        ft_ns
    )

    diffs.append({
        # Positive = fine-tuned worse
        "wer_difference":
            ft_wer - base_wer,

        # Negative = fine-tuned better
        "cer_difference":
            ft_cer - base_cer,

        "cer_no_spaces_difference":
            ft_cer_ns - base_cer_ns,
    })


diffs = pd.DataFrame(
    diffs
)

print(
    "Fine-tuned minus baseline "
    "(percentage points)\n"
)

for col in diffs.columns:

    values = (
        diffs[col] * 100
    )

    lo, hi = np.percentile(
        values,
        [2.5, 97.5]
    )

    print(
        col,
        "\n mean:",
        round(values.mean(), 2),
        "\n 95% CI:",
        (
            round(lo, 2),
            round(hi, 2),
        ),
        "\n"
    )

Fine-tuned minus baseline (percentage points)

wer_difference 
 mean: 3.77 
 95% CI: (np.float64(1.13), np.float64(6.63)) 

cer_difference 
 mean: -3.0 
 95% CI: (np.float64(-3.97), np.float64(-2.05)) 

cer_no_spaces_difference 
 mean: -5.65 
 95% CI: (np.float64(-6.72), np.float64(-4.59)) 



In [ ]:
# Cell 28 — Save paired bootstrap final summary

import pandas as pd

paired_bootstrap_summary = pd.DataFrame([
    {
        "metric": "WER",
        "finetuned_minus_baseline_pp": 3.77,
        "ci_low_pp": 1.13,
        "ci_high_pp": 6.63,
        "interpretation": "MMS zero-shot baseline better"
    },
    {
        "metric": "CER",
        "finetuned_minus_baseline_pp": -3.00,
        "ci_low_pp": -3.97,
        "ci_high_pp": -2.05,
        "interpretation": "Fine-tuned MMS better"
    },
    {
        "metric": "CER_no_spaces",
        "finetuned_minus_baseline_pp": -5.65,
        "ci_low_pp": -6.72,
        "ci_high_pp": -4.59,
        "interpretation": "Fine-tuned MMS better"
    },
])

out_path = (
    RESULTS_ROOT
    / "paired_bootstrap_baseline_vs_best_finetuned.csv"
)

paired_bootstrap_summary.to_csv(
    out_path,
    index=False,
    encoding="utf-8"
)

print(paired_bootstrap_summary.to_string(index=False))
print("\n✅ Saved:", out_path)

       metric  finetuned_minus_baseline_pp  ci_low_pp  ci_high_pp                interpretation
          WER                         3.77       1.13        6.63 MMS zero-shot baseline better
          CER                        -3.00      -3.97       -2.05         Fine-tuned MMS better
CER_no_spaces                        -5.65      -6.72       -4.59         Fine-tuned MMS better

✅ Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/paired_bootstrap_baseline_vs_best_finetuned.csv


In [ ]:
# Cell 1 — Mount Google Drive and define project paths

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
    / "final_test_v1_3"
)

print("Project exists:", PROJECT_ROOT.exists())
print("Results exists:", RESULTS_ROOT.exists())
print("Results path:", RESULTS_ROOT)

Mounted at /content/drive
Project exists: True
Results exists: True
Results path: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3


In [ ]:
# Cell 2 — Check all important final-test result folders and files

important_files = {

    "Final evaluation manifest":
        PROJECT_ROOT
        / "data/metadata/segments_metadata_v1_3_final_eval.csv",

    "MMS baseline predictions":
        RESULTS_ROOT
        / "mms_greedy_baseline/predictions.csv",

    "MMS baseline summary":
        RESULTS_ROOT
        / "mms_greedy_baseline/summary.json",

    "MMS half-Bible predictions":
        RESULTS_ROOT
        / "mms_half_bible_specaug/predictions.csv",

    "MMS half-Bible summary":
        RESULTS_ROOT
        / "mms_half_bible_specaug/summary.json",

    "MMS full predictions":
        RESULTS_ROOT
        / "mms_full_specaug/predictions.csv",

    "Fadhma predictions":
        RESULTS_ROOT
        / "fadhma_full/predictions.csv",

    "OmniASR predictions":
        RESULTS_ROOT
        / "omni_full/predictions.csv",

    "Whisper predictions":
        RESULTS_ROOT
        / "whisper_small_zero_shot/predictions.csv",

    "Whisper summary":
        RESULTS_ROOT
        / "whisper_small_zero_shot/summary.json",

    "Baseline vs fine-tuned comparison":
        RESULTS_ROOT
        / "baseline_vs_best_finetuned.csv",

    "Bootstrap confidence intervals":
        RESULTS_ROOT
        / "bootstrap_95ci_final_test.csv",

    "Baseline deletion examples":
        RESULTS_ROOT
        / "baseline_deletion_examples.csv",

    "Fine-tuned deletion examples":
        RESULTS_ROOT
        / "finetuned_deletion_examples.csv",
}


print("=" * 90)

missing = []

for name, path in important_files.items():

    exists = path.exists()

    symbol = "✅" if exists else "❌"

    print(
        f"{symbol} {name:40s} "
        f"{path}"
    )

    if not exists:
        missing.append(name)

print("=" * 90)

if missing:
    print("\nMissing:")
    for x in missing:
        print("-", x)
else:
    print("\n✅ All expected saved files are present.")

✅ Final evaluation manifest                /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/segments_metadata_v1_3_final_eval.csv
✅ MMS baseline predictions                 /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/mms_greedy_baseline/predictions.csv
✅ MMS baseline summary                     /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/mms_greedy_baseline/summary.json
✅ MMS half-Bible predictions               /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/mms_half_bible_specaug/predictions.csv
✅ MMS half-Bible summary                   /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/mms_half_bible_specaug/summary.json
✅ MMS full predictions                     /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/final_test_v1_3/mms_full_specaug/predictions.csv
✅ Fadhma predictions       